<a href="https://colab.research.google.com/github/FivaKit/practice/blob/main/hw_language_modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p style="align: center;"><img src="https://static.tildacdn.com/tild6636-3531-4239-b465-376364646465/Deep_Learning_School.png" width="400"></p>

# Домашнее задание. Обучение языковой модели с помощью LSTM (10 баллов)

Э
В этом задании Вам предстоит обучить языковую модель с помощью рекуррентной нейронной сети. В отличие от семинарского занятия, Вам необходимо будет работать с отдельными словами, а не буквами.


Установим модуль ```datasets```, чтобы нам проще было работать с данными.

In [1]:
!pip install datasets

Импорт необходимых библиотек

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from datasets import load_dataset
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.model_selection import train_test_split
import nltk

from collections import Counter
from typing import List

import seaborn
seaborn.set(palette='summer')

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

## Подготовка данных

Воспользуемся датасетом imdb. В нем хранятся отзывы о фильмах с сайта imdb. Загрузим данные с помощью функции ```load_dataset```

In [5]:
# Загрузим датасет
dataset = load_dataset('imdb')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

HfUriError: Invalid HF URI 'hf://datasets/imdb@e6281661ce1c48d982bc483cf8a173c1bbeb5d31/.huggingface.yaml'. Repository id must be 'namespace/name', got 'imdb'.

### Препроцессинг данных и создание словаря (1 балл)

Далее вам необходмо самостоятельно произвести препроцессинг данных и получить словарь или же просто ```set``` строк. Что необходимо сделать:

1. Разделить отдельные тренировочные примеры на отдельные предложения с помощью функции ```sent_tokenize``` из бибилиотеки ```nltk```. Каждое отдельное предложение будет одним тренировочным примером.
2. Оставить только те предложения, в которых меньше ```word_threshold``` слов.
3. Посчитать частоту вхождения каждого слова в оставшихся предложениях. Для деления предлоения на отдельные слова удобно использовать функцию ```word_tokenize```.
4. Создать объект ```vocab``` класса ```set```, положить в него служебные токены '\<unk\>', '\<bos\>', '\<eos\>', '\<pad\>' и vocab_size самых частовстречающихся слов.   

In [ ]:
sentences = []
word_threshold = 32
for sentence in dataset['train']['text']:
  sentences.extend([x.lower() for x in sent_tokenize(sentence, 'english') if len(word_tokenize(x))<word_threshold])
# Получить отдельные предложения и поместить их в sentences

In [ ]:
print("Всего предложений:", len(sentences))

Посчитаем для каждого слова его встречаемость.

In [ ]:
words = Counter()
for sentence in sentences:
  for word in word_tokenize(sentence, 'english'):
    words[word]+=1

# Расчет встречаемости слов

In [ ]:
len(words)

In [ ]:
common_w = words.most_common(40000)
common_w = [x[0] for x in common_w]
common_w[:10]

In [ ]:
common_w

Добавим в словарь ```vocab_size``` самых встречающихся слов.

In [ ]:
vocab = set(['<unk>', '<bos>', '<eos>', '<pad>'])
for x in common_w: vocab.add(x)
vocab_size = 40000


In [ ]:
assert '<unk>' in vocab
assert '<bos>' in vocab
assert '<eos>' in vocab
assert '<pad>' in vocab
assert len(vocab) == vocab_size + 4

In [ ]:
print("Всего слов в словаре:", len(vocab))

### Подготовка датасета (1 балл)

Далее, как и в семинарском занятии, подготовим датасеты и даталоадеры.

В классе ```WordDataset``` вам необходимо реализовать метод ```__getitem__```, который будет возвращать сэмпл данных по входному idx, то есть список целых чисел (индексов слов).

Внутри этого метода необходимо добавить служебные токены начала и конца последовательности, а также токенизировать соответствующее предложение с помощью ```word_tokenize``` и сопоставить ему индексы из ```word2ind```.

In [ ]:
word2ind = {char: i for i, char in enumerate(vocab)}
ind2word = {i: char for char, i in word2ind.items()}

In [ ]:
class WordDataset:
    def __init__(self, sentences):
        self.data = sentences
        self.unk_id = word2ind['<unk>']
        self.bos_id = word2ind['<bos>']
        self.eos_id = word2ind['<eos>']
        self.pad_id = word2ind['<pad>']

    def __getitem__(self, idx: int) -> List[int]:
        # tokenized_sentence = []
        # tokenized_sentence+=[self.bos_id]
        # for word in word_tokenize(self.data[idx]):
        #   tokenized_sentence.append(word2ind.get(word, self.unk_id))
        # tokenized_sentence.append(self.eos_id)
        sentence = self.data[idx] # Возвращаем сэмпл данных по входному индексу
        tokenized_sentence = word_tokenize(sentence) # Делим предложение на токены
        tokenized_sentence = [self.bos_id] + [word2ind.get(word, self.unk_id) for word in tokenized_sentence] + [self.eos_id] # Вставляем в начало bos и в конце eos, а также вставляем unk в случае отсутсвия слова в словаре
        return tokenized_sentence

    def __len__(self) -> int:
        return len(self.data)

In [ ]:
def collate_fn_with_padding(
    input_batch: List[List[int]], pad_id=word2ind['<pad>']) -> torch.Tensor:
    seq_lens = [len(x) for x in input_batch]
    max_seq_len = max(seq_lens)

    new_batch = []
    for sequence in input_batch:
        for _ in range(max_seq_len - len(sequence)):
            sequence.append(pad_id)
        new_batch.append(sequence)

    sequences = torch.LongTensor(new_batch).to(device)

    new_batch = {
        'input_ids': sequences[:,:-1],
        'target_ids': sequences[:,1:]
    }

    return new_batch

In [ ]:
train_sentences, eval_sentences = train_test_split(sentences, test_size=0.2)
eval_sentences, test_sentences = train_test_split(eval_sentences, test_size=0.5)

train_dataset = WordDataset(train_sentences)
eval_dataset = WordDataset(eval_sentences)
test_dataset = WordDataset(test_sentences)

batch_size = 128

train_dataloader = DataLoader(
    train_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

eval_dataloader = DataLoader(
    eval_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

test_dataloader = DataLoader(
    test_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

## Обучение и архитектура модели

Вам необходимо на практике проверить, что влияет на качество языковых моделей. В этом задании нужно провести серию экспериментов с различными вариантами языковых моделей и сравнить различия в конечной перплексии на тестовом множестве.

Возмоэные идеи для экспериментов:

* Различные RNN-блоки, например, LSTM или GRU. Также можно добавить сразу несколько RNN блоков друг над другом с помощью аргумента num_layers. Вам поможет официальная документация [здесь](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html)
* Различные размеры скрытого состояния. Различное количество линейных слоев после RNN-блока. Различные функции активации.
* Добавление нормализаций в виде Dropout, BatchNorm или LayerNorm
* Различные аргументы для оптимизации, например, подбор оптимального learning rate или тип алгоритма оптимизации SGD, Adam, RMSProp и другие
* Любые другие идеи и подходы

После проведения экспериментов необходимо составить таблицу результатов, в которой описан каждый эксперимент и посчитана перплексия на тестовом множестве.

Учтите, что эксперименты, которые различаются, например, только размером скрытого состояния или количеством линейных слоев считаются, как один эксперимент.

Успехов!

### Функция evaluate (1 балл)

Заполните функцию ```evaluate```

In [ ]:
b = next(iter(train_dataloader))
b['input_ids']

In [ ]:
def evaluate(model, criterion, dataloader) -> float:
    model.eval()
    perplexity = []
    with torch.no_grad():
        for batch in dataloader:
            logits = model(batch['input_ids']).flatten(start_dim=0, end_dim=1)
            loss = criterion(logits, batch['target_ids'].flatten())
            perplexity.append(torch.exp(loss).item())

        perplexity = sum(perplexity) / len(perplexity)

    return perplexity

### Train loop (1 балл)

Напишите функцию для обучения модели.

In [ ]:
def train_model(model, criterion, optimizer, epochs):
  losses = []
  perplexities = []
  for epoch in tqdm(range(epochs)):
    model.train()
    loss_epoch = []
    for batch in tqdm(train_dataloader):
      optimizer.zero_grad()
      logits = model(batch['input_ids']).flatten(start_dim=0, end_dim=1)
      loss = criterion(logits, batch['target_ids'].flatten())
      loss.backward()
      optimizer.step()
      loss_epoch.append(loss.item())
    losses.append(sum(loss_epoch)/len(loss_epoch))
    perplexities.append(evaluate(model, criterion, eval_dataloader))
  return losses, perplexities


### Первый эксперимент (2 балла)

Определите архитектуру модели и обучите её.

In [ ]:
class LanguageModel(nn.Module):
    def __init__(self, hidden_dim, vocab_size):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, hidden_dim)
        self.rnn = nn.LSTM(hidden_dim, hidden_dim)
        self.linear = nn.Linear(hidden_dim, hidden_dim)
        self.proection = nn.Linear(hidden_dim, vocab_size)
        self.nonlin = nn.Tanh()
        self.dropout = nn.Dropout(0.1)
    def forward(self, input_batch: torch.Tensor) -> torch.Tensor:
        output, _ = self.rnn(self.embeddings(input_batch))
        output = self.dropout(self.linear(self.nonlin(output)))
        output = self.proection(self.nonlin(output))
        return output

In [ ]:
model = LanguageModel(hidden_dim=256, vocab_size=len(vocab)).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=word2ind['<pad>'])
optimizer = torch.optim.Adam(model.parameters())

In [ ]:
losses, perplexities = train_model(model=model, criterion=criterion, optimizer=optimizer, epochs=5)

In [ ]:
plt.plot(np.arange(len(losses)), losses)
plt.title('Losses')
plt.xlabel("epoch")
plt.show()

In [ ]:
print(min(perplexities))
plt.plot(np.arange(len(perplexities)), perplexities)
plt.title('Perplexity')
plt.xlabel("epoch")
plt.show()

In [ ]:
len(eval_dataloader)

### Второй эксперимент (2 балла)

Попробуйте что-то поменять в модели или в пайплайне обучения, идеи для экспериментов можно подсмотреть выше.

In [ ]:
class LanguageModel2(nn.Module):
    def __init__(self, hidden_dim, vocab_size):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, hidden_dim)
        self.rnn = nn.LSTM(hidden_dim, hidden_dim)
        self.linear = nn.Linear(hidden_dim, hidden_dim)
        self.proection = nn.Linear(hidden_dim, vocab_size)
        self.nonlin = nn.Tanh()
        self.dropout = nn.Dropout(0.1)
    def forward(self, input_batch: torch.Tensor) -> torch.Tensor:
        output, _ = self.rnn(self.embeddings(input_batch))
        output = self.dropout(self.linear(self.nonlin(output)))
        output = self.proection(self.nonlin(output))
        return output

In [ ]:
def evaluate2(model, criterion, dataloader, scheduler) -> float:
    model.eval()
    perplexity = []
    with torch.no_grad():
        for batch in dataloader:
            logits = model(batch['input_ids']).flatten(start_dim=0, end_dim=1)
            loss = criterion(logits, batch['target_ids'].flatten())
            scheduler.step(loss)
            perplexity.append(torch.exp(loss).item())

        perplexity = sum(perplexity) / len(perplexity)

    return perplexity

In [ ]:
def train_model2(model, criterion, optimizer, scheduler, epochs):
  losses = []
  perplexities = []
  norms = []
  for epoch in tqdm(range(epochs)):
    model.train()
    norms_epoch = []
    loss_epoch = []
    for batch in tqdm(train_dataloader):
      optimizer.zero_grad()
      logits = model(batch['input_ids']).flatten(start_dim=0, end_dim=1)
      loss = criterion(logits, batch['target_ids'].flatten())
      loss.backward()
      grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
      norms_epoch.append(grad_norm)
      optimizer.step()
      loss_epoch.append(loss.item())
    losses.append(sum(loss_epoch)/len(loss_epoch))
    perplexities.append(evaluate2(model, criterion, eval_dataloader, scheduler))
    norms.append(sum(norms_epoch)/len(norms_epoch))
  return losses, perplexities, norms

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [ ]:
model = LanguageModel2(hidden_dim=512, vocab_size=len(vocab)).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=word2ind['<pad>'])
optimizer = torch.optim.Adam(model.parameters())
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.9,
    patience=5
)

In [ ]:
losses, perplexities, norms = train_model2(model=model, criterion=criterion, optimizer=optimizer, scheduler=scheduler, epochs=15)

In [ ]:
plt.plot(np.arange(len(losses)), losses)
plt.title('Losses')
plt.xlabel("epoch")
plt.show()

In [ ]:
print(min(perplexities))
plt.plot(np.arange(len(perplexities)), perplexities)
plt.title('Perplexity')
plt.xlabel("epoch")
plt.show()


In [ ]:
norms = np.array([i.item() for i in norms])
norms

In [ ]:
with torch.no_grad():
  plt.plot(np.arange(len(norms)), norms)
  plt.title('Norms')
  plt.xlabel("epoch")
  plt.show()

### Отчет (2 балла)

Опишите проведенные эксперименты. Сравните перплексии полученных моделей. Предложите идеи по улучшению качества моделей.